# Aula 2 — Ambientes Conda e BLAST

Disciplina: **EQM — Bioinformática e Biologia Molecular**

Nesta aula, além do BLAST, vamos estabelecer a forma de trabalho que será reutilizada
durante a disciplina:

**runtime → Conda → Google Drive → diretório do projeto → arquivo de entrada → análise → arquivo de saída**

A partir daqui, cada notebook começa localizando o projeto e verificando o que foi produzido
na aula anterior.

## 0. Preparar o runtime

O Google Drive guarda os **dados** entre as aulas, mas o runtime do Colab é temporário.
Programas instalados no runtime podem desaparecer quando a sessão termina.

Por isso, quando uma aula precisar de ferramentas externas, começaremos verificando se
o Conda já existe. Se não existir, ele será instalado antes de qualquer outra configuração.

> Esta deve ser a primeira célula executável do notebook, porque a instalação do Conda
> pode reiniciar o runtime.

In [ ]:
import shutil

if shutil.which("conda"):
    print("Conda já está disponível neste runtime.")
else:
    !pip install -q condacolab
    import condacolab
    condacolab.install()

### Verificar o Conda e configurar Bioconda

Usaremos a configuração recomendada pelo Bioconda: `conda-forge` com maior prioridade,
seguido de `bioconda`, e prioridade estrita.

Como `conda config --add` adiciona canais do menor para o maior nível de prioridade,
executamos primeiro `bioconda` e depois `conda-forge`.

In [ ]:
!conda --version
!conda config --remove-key channels 2>/dev/null || true
!conda config --add channels bioconda
!conda config --add channels conda-forge
!conda config --set channel_priority strict
!conda config --show channels

## 1. Retomar o projeto no Google Drive

Todas as práticas usam a mesma raiz:

`/content/drive/MyDrive/Bioinformatica_Biologia_Molecular`

Os resultados de uma aula são lidos pela aula seguinte. Assim, os **dados persistem**
mesmo quando o runtime do Colab é encerrado.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Criar e reconhecer os diretórios preestabelecidos

Nesta aula criamos a estrutura principal de diretórios do curso. Nas aulas seguintes
não inventaremos novos caminhos: apenas retomaremos estes mesmos diretórios.

In [ ]:
from pathlib import Path
import os

ROOT = Path("/content/drive/MyDrive/Bioinformatica_Biologia_Molecular")
RUN = "SRR15736591"
SAMPLE = "hypochilus_petrunkevitchi_SRR15736591"

PASTAS = {
    "01_bancos": ROOT / "01_bancos",
    "02_blast": ROOT / "02_blast",
    "03_raw": ROOT / "03_sra_fastq" / "raw-fastq",
    "04_qc": ROOT / "04_qc_trimming",
    "04_trimmed": ROOT / "04_qc_trimming" / "trimmed",
    "05_assemblies": ROOT / "05_spades" / "spades-assemblies",
    "05_contigs": ROOT / "05_spades" / "spades-assemblies" / "contigs",
    "06_match": ROOT / "06_uce_match",
    "06_probes": ROOT / "06_uce_match" / "probes",
    "06_results": ROOT / "06_uce_match" / "uce-search-results",
    "07_taxon_sets": ROOT / "07_uce_extract" / "taxon-sets" / "all",
    "08_integracao": ROOT / "08_integracao",
    "ambientes": ROOT / "ambientes",
}

for pasta in PASTAS.values():
    pasta.mkdir(parents=True, exist_ok=True)

os.chdir(ROOT)

print("Diretório atual:", Path.cwd())
print("\nEstrutura principal do projeto:")
for chave, pasta in PASTAS.items():
    print(f"{chave:15s} -> {pasta.relative_to(ROOT)}")

In [ ]:
print("\nPastas existentes na raiz:")
for p in sorted(ROOT.iterdir()):
    if p.is_dir():
        print(" -", p.name)

## 2. Verificar a entrada da aula anterior

A Aula 1 salvou a sequência `MK270576.1.fasta` em `01_bancos/`.
Agora vamos usar **esse mesmo arquivo**, em vez de baixá-lo novamente.

In [ ]:
INPUT_FASTA = PASTAS["01_bancos"] / "MK270576.1.fasta"
OUT = PASTAS["02_blast"]

print("Entrada :", INPUT_FASTA)
print("Saída   :", OUT)

if not INPUT_FASTA.exists():
    raise FileNotFoundError(
        "MK270576.1.fasta não foi encontrado em 01_bancos. "
        "Execute primeiro a prática de Bancos biológicos."
    )

print("\nArquivo encontrado. Primeiras linhas:")
print(INPUT_FASTA.read_text()[:400])

## 3. Criar o ambiente `bioinfo`

No computador pessoal, um ambiente Conda pode permanecer instalado por muito tempo.
No Colab, o runtime é temporário. Portanto, verificamos o ambiente a cada sessão.

Nesta aula instalaremos somente o que precisamos agora: **BLAST**.

In [ ]:
!conda env list | grep -qE '^bioinfo[[:space:]]' || conda create -y -n bioinfo python=3.11
!conda install -y -n bioinfo blast
!conda run -n bioinfo blastn -version

### Demonstrar ativação

Em um terminal convencional, usaríamos `conda activate bioinfo`.
No Colab, a ativação dentro de uma célula shell não permanece nas células seguintes.
Por isso, entre células usamos `conda run -n bioinfo ...`.

In [ ]:
%%bash
source "$(conda info --base)/etc/profile.d/conda.sh"
conda activate bioinfo
echo "Ambiente ativo nesta célula: $CONDA_DEFAULT_ENV"
which blastn
blastn -version

## 4. Executar BLASTn

A sequência da Aula 1 é a **query**. A busca será feita remotamente contra `nt`.

In [ ]:
saida = OUT / "blastn_nt.tsv"

cmd = (
    f"blastn -query '{INPUT_FASTA}' -db nt -remote "
    f"-max_target_seqs 15 "
    f'-outfmt "6 qseqid sacc pident length qcovs evalue bitscore stitle" '
    f"-out '{saida}'"
)

!conda run -n bioinfo bash -c "$cmd"

print("Resultado salvo em:", saida)

## 5. Ler e interpretar a saída

In [ ]:
import pandas as pd

cols = [
    "query", "accession_hit", "identity_pct", "alignment_length",
    "query_coverage_pct", "evalue", "bitscore", "title"
]
df = pd.read_csv(saida, sep="\t", names=cols)
df.head(15)

In [ ]:
df[
    ["accession_hit", "identity_pct", "query_coverage_pct", "evalue", "bitscore", "title"]
].head(10)

Perguntas:
- o primeiro hit também apresenta alta cobertura?
- identidade alta com cobertura baixa é suficiente?
- o título do registro prova sozinho a identificação?
- qual a diferença entre similaridade e identificação?

## 6. Registrar o ambiente usado nesta aula

In [ ]:
ENV_FILE = PASTAS["ambientes"] / "aula02_bioinfo.yml"
!conda env export -n bioinfo --from-history > "$ENV_FILE"
print("Especificação salva em:", ENV_FILE)

## Saída que continuará no projeto

A próxima aula não depende biologicamente do BLAST para baixar o SRA, mas o arquivo fica
registrado no mesmo projeto:

`02_blast/blastn_nt.tsv`

Na Aula 3 começa o pipeline principal do dataset de *Hypochilus*:

**SRA → FASTQ → QC → trimming → montagem → UCEs**